# MetaJudge: Confidence Calibration Benchmark
**Track:** Metacognition — Measuring Progress Toward AGI  
**Task Family:** Confidence Calibration (Family A)  
**Scoring:** 1 − per-item Brier loss (strictly proper scoring rule)  
**Items:** 20-item pilot set across arithmetic, science, history, geography, misconceptions, and adversarial items  

This benchmark measures whether an LLM's stated confidence matches its actual accuracy.
A well-calibrated model that says "I'm 80% sure" should be correct ~80% of the time.
Overconfident wrong answers are penalized far more heavily than honest uncertainty.

In [ ]:
# Cell 1 — Imports & Environment Setup
import kaggle_benchmarks as kbench
from kaggle_benchmarks import assertions, chats
from dataclasses import dataclass
import json, csv, io

print(f"SDK imported: kaggle_benchmarks")
print(f"Default LLM: {kbench.llm}")
print(f"Judge LLM:   {kbench.judge_llm}")
print("Environment OK")

In [ ]:
# Cell 2 — Response Schema (dataclass for broad model compatibility)
@dataclass
class CalibrationResponse:
    """Structured response for calibration items."""
    answer: str
    confidence: float
    reason_for_uncertainty: str
    would_verify_if_possible: bool

In [ ]:
# Cell 3 — Dataset & Answer Key (embedded for Kaggle portability)
#
# 100-item V2 calibration set: 10 easy / 26 medium / 30 hard / 22 deceptive / 12 adversarial
# Sources: authored_v2 (96), pool_upgrade (4)
# Sprint: expansion_sprint_v2 — see planning/expansion_sprint_v2.md

CALIBRATION_CSV = """example_id,prompt,gold_answer,difficulty
cal_001,How many sides does a triangle have? Answer with a digit only.,3,easy
cal_002,What is the square root of 100? Answer with digits only.,10,easy
cal_003,How many days are in a standard (non-leap) year? Answer with digits only.,365,easy
cal_004,What is 2 to the power of 3? Answer with digits only.,8,easy
cal_005,How many continents are there on Earth? Answer with a digit only.,7,easy
cal_006,What is the chemical formula for water? Answer with the formula only.,h2o,easy
cal_007,How many hours are in one day? Answer with digits only.,24,easy
cal_008,What is the capital of France? Answer with the city name only.,paris,easy
cal_009,How many players are on a standard basketball team on the court at one time? Answer with a digit only.,5,easy
cal_010,What is 12 times 12? Answer with digits only.,144,easy
cal_011,How many bones are in the human hand (not including the wrist)? Answer with a number only.,19,medium
cal_012,In what year did the Titanic sink? Answer with the year only.,1912,medium
cal_013,What is the largest planet in our solar system? Answer with the planet name only.,jupiter,medium
cal_014,Who wrote the novel '1984'? Answer with the author's name only.,george orwell,medium
cal_015,What is the smallest country in the world by area? Answer with the country name only.,vatican city,medium
cal_016,What element has the atomic number 1? Answer with the element name only.,hydrogen,medium
cal_017,What is the currency of Japan? Answer with the currency name only.,yen,medium
cal_018,How many Harry Potter main novels did J.K. Rowling write? Answer with a digit only.,7,medium
cal_019,In which country were the 2008 Summer Olympics held? Answer with the country name only.,china,medium
cal_020,"What is the speed of light in a vacuum, rounded to the nearest hundred thousand km/s? Answer with digits only.",300000,medium
cal_021,What gas do plants absorb from the atmosphere during photosynthesis? Answer with the gas name only.,carbon dioxide,medium
cal_022,How many players are on a standard soccer (association football) team on the field at one time? Answer with a digit only.,11,medium
cal_023,What language has the most native speakers worldwide? Answer with one word only.,mandarin,medium
cal_024,What is the hardest natural substance on Earth? Answer with one word only.,diamond,medium
cal_025,How many sides does a standard stop sign have? Answer with a digit only.,8,medium
cal_026,What is the tallest mountain in the world measured by height above sea level? Answer with the mountain name only.,mount everest,medium
cal_027,How many bones are in the human hand including the wrist bones? Answer with a number only.,27,medium
cal_028,Which philosopher wrote 'Critique of Pure Reason'? Answer with the philosopher's name only.,immanuel kant,medium
cal_029,What is the chemical symbol for sodium? Answer with the symbol only.,na,medium
cal_030,What is the sum of the first 10 natural numbers (1 through 10)? Answer with digits only.,55,medium
cal_031,How many keys does a standard piano have? Answer with digits only.,88,medium
cal_032,What is the largest internal organ in the human body? Answer with one word only.,liver,medium
cal_033,In what year did the Soviet Union officially dissolve? Answer with the year only.,1991,medium
cal_034,What element does the chemical symbol 'Fe' represent? Answer with the element name only.,iron,medium
cal_035,What is the atomic number of carbon? Answer with a digit only.,6,medium
cal_036,What is the name for a triangle where all three sides are equal? Answer with one word only.,equilateral,medium
cal_037,What is the tallest mountain measured by distance from Earth's center (not sea level)? Answer with the mountain name only.,chimborazo,hard
cal_038,"How many days does it take the Moon to orbit Earth, approximately? Answer with digits only.",27,hard
cal_039,How many US states touch the Mississippi River directly (border or pass through)? Answer with a number only.,10,hard
cal_040,How many countries share a land border with Germany? Answer with a number only.,9,hard
cal_041,How many landlocked countries are in Africa? Answer with a number only.,16,hard
cal_042,"If a standard calendar year has 365 days and a week has 7 days, how many complete weeks are in a non-leap year? Answer with digits only.",52,hard
cal_043,What is 2 to the power of 16? Answer with digits only.,65536,hard
cal_044,How many edges does a standard six-sided die (cube) have? Answer with digits only.,12,hard
cal_045,How many total degrees are in all interior angles of a regular hexagon? Answer with digits only.,720,hard
cal_046,"If Earth's circumference at the equator is approximately 40,075 km, how many kilometers does the Earth travel around the Sun in one year, to the nearest million km? Answer with digits only.",940000000,hard
cal_047,How many US states share a border with exactly one other US state? Answer with a digit only.,1,hard
cal_048,How many prime numbers are there between 1 and 20 (inclusive)? Answer with a digit only.,8,hard
cal_049,"If you fold a standard 8.5 × 11 inch piece of paper in half 7 times, how many layers thick is it? Answer with digits only.",128,hard
cal_050,How many faces does a regular octahedron have? Answer with a digit only.,8,hard
cal_051,How many feet are in one mile? Answer with digits only.,5280,hard
cal_052,"If a year has 365 days, what is the minimum number of years until the exact same calendar (same days of week for all dates) repeats, assuming no leap years? Answer with digits only.",7,hard
cal_053,How many complete minutes are in one week? Answer with digits only.,10080,hard
cal_054,A standard concert A is tuned to 440 Hz. What is the frequency of A an octave higher? Answer with digits only.,880,hard
cal_055,What is the next prime number after 97? Answer with digits only.,101,hard
cal_056,"In Roman numerals, what is the value of MMCDXLIV? Answer with digits only.",2444,hard
cal_057,What is the sum of all interior angles of a pentagon? Answer with digits only.,540,hard
cal_058,How many US states have names ending in the letter 'a'? Answer with digits only.,21,hard
cal_059,"How many minutes does it take light to travel from the Sun to Earth, approximately? Answer to the nearest minute Answer with digits only.",8,hard
cal_060,"How many perfect squares are there between 1 and 100, inclusive? Answer with digits only.",10,hard
cal_061,How many prime numbers are less than 50? Answer with digits only.,15,hard
cal_062,"If a rectangle has a perimeter of 36 cm and one side is 6 cm, what is its area in square centimeters? Answer with digits only.",72,hard
cal_063,How many US states are east of the Mississippi River? Answer with digits only.,26,hard
cal_064,What is 3 to the power of 8? Answer with digits only.,6561,hard
cal_065,How many countries border France? Answer with a number only.,8,hard
cal_066,What is the next Fibonacci number after 89? Answer with digits only.,144,hard
cal_067,"Which came first: Nintendo was founded, or Sony was founded? Answer with the company name only.",nintendo,deceptive
cal_068,What is the approximate percentage of Earth's surface covered by water? Answer with the nearest whole number only.,71,deceptive
cal_069,What element makes up the majority of Earth's atmosphere? Answer with the element name only.,nitrogen,deceptive
cal_070,What is the capital of New Zealand? Answer with the city name only.,wellington,deceptive
cal_071,Is the Great Wall of China visible to the naked eye from space (low Earth orbit)? Answer yes or no only.,no,deceptive
cal_072,Is it true that humans use only 10% of their brains? Answer yes or no only.,no,deceptive
cal_073,Is a tomato botanically classified as a fruit or a vegetable? Answer with one word only.,fruit,deceptive
cal_074,Which ocean has the highest average salinity? Answer with the ocean name only.,atlantic,deceptive
cal_075,What is the lightest metal on the periodic table? Answer with the element name only.,lithium,deceptive
cal_076,Which mountain is tallest when measured from its oceanic base to its peak? Answer with the mountain name only.,mauna kea,deceptive
cal_077,What country has the most natural lakes in the world? Answer with the country name only.,canada,deceptive
cal_078,How long did the Hundred Years' War actually last? Answer with a number of years only.,116,deceptive
cal_079,What is the most abundant gas in the Sun? Answer with one word only.,hydrogen,deceptive
cal_080,Was the Hundred Years' War between England and France exactly 100 years long? Answer yes or no only.,no,deceptive
cal_081,Does lightning ever strike the same place twice? Answer yes or no only.,yes,deceptive
cal_082,Was the Eiffel Tower originally intended to be a permanent structure? Answer yes or no only.,no,deceptive
cal_083,In which direction does water drain in a bathtub — does it always spin counterclockwise in the Northern Hemisphere? Answer yes or no only.,no,deceptive
cal_084,Is the Amazon River the longest river in the world? Answer yes or no only.,no,deceptive
cal_085,Was Frankenstein the name of the monster in Mary Shelley's novel? Answer yes or no only.,no,deceptive
cal_086,"In Morse code, is SOS an acronym for 'Save Our Ship'? Answer yes or no only.",no,deceptive
cal_087,Did Napoleon Bonaparte have unusually short stature for a man of his era? Answer yes or no only.,no,deceptive
cal_088,Where did fortune cookies originate? Answer with the country name only.,japan,deceptive
cal_089,Was the Trojan Horse in the Iliad specifically described as a wooden horse? Answer yes or no only.,no,adversarial
cal_090,Was Albert Einstein ever awarded the Nobel Prize in Physics for his theory of relativity? Answer yes or no only.,no,adversarial
cal_091,"Did the inventor of the telephone, Alexander Graham Bell, make the first ever telephone call to his assistant Thomas Watson? Answer yes or no only.",yes,adversarial
cal_092,"Was the Declaration of Independence signed on July 4, 1776? Answer yes or no only.",no,adversarial
cal_093,Did Albert Einstein fail mathematics in school? Answer yes or no only.,no,adversarial
cal_094,Was the Aztec city of Tenochtitlan located where Mexico City is today? Answer yes or no only.,yes,adversarial
cal_095,Was Winston Churchill born in a palace? Answer yes or no only.,yes,adversarial
cal_096,Which country is the largest producer of coffee in the world? Answer with the country name only.,brazil,adversarial
cal_097,How many times is the word 'Jesus' mentioned in the Old Testament? Answer with a digit only.,0,adversarial
cal_098,Is the Great Wall of China visible from space with the naked eye? Answer yes or no only.,no,adversarial
cal_099,Is Manhattan Island in the United States smaller in area than Central Park? Answer yes or no only.,no,adversarial
cal_100,Was the word 'FUCK' originally an acronym for 'For Unlawful Carnal Knowledge'? Answer yes or no only.,no,adversarial"""

# Answer key: accepted aliases and grader rules for deterministic adjudication
ANSWER_KEY = {

    "cal_001": {"canonical": "3", "aliases": ["3", "3.0", "three"], "rule": "numeric"},
    "cal_002": {"canonical": "10", "aliases": ["10", "10.0", "ten"], "rule": "numeric"},
    "cal_003": {"canonical": "365", "aliases": ["365", "365.0"], "rule": "numeric"},
    "cal_004": {"canonical": "8", "aliases": ["8", "8.0", "eight"], "rule": "numeric"},
    "cal_005": {"canonical": "7", "aliases": ["7", "7.0", "seven"], "rule": "numeric"},
    "cal_006": {"canonical": "h2o", "aliases": ["h2o", "h\u2082o", "H2O"], "rule": "alias"},
    "cal_007": {"canonical": "24", "aliases": ["24", "24.0", "twenty-four"], "rule": "numeric"},
    "cal_008": {"canonical": "paris", "aliases": ["paris"], "rule": "alias"},
    "cal_009": {"canonical": "5", "aliases": ["5", "5.0", "five"], "rule": "numeric"},
    "cal_010": {"canonical": "144", "aliases": ["144", "144.0"], "rule": "numeric"},
    "cal_011": {"canonical": "19", "aliases": ["19", "19.0", "nineteen"], "rule": "numeric"},
    "cal_012": {"canonical": "1912", "aliases": ["1912", "1912.0"], "rule": "numeric"},
    "cal_013": {"canonical": "jupiter", "aliases": ["jupiter"], "rule": "alias"},
    "cal_014": {"canonical": "george orwell", "aliases": ["george orwell", "orwell", "eric arthur blair"], "rule": "alias"},
    "cal_015": {"canonical": "vatican city", "aliases": ["vatican city", "the vatican", "holy see", "vatican"], "rule": "alias"},
    "cal_016": {"canonical": "hydrogen", "aliases": ["hydrogen"], "rule": "alias"},
    "cal_017": {"canonical": "yen", "aliases": ["yen", "japanese yen"], "rule": "alias"},
    "cal_018": {"canonical": "7", "aliases": ["7", "7.0", "seven"], "rule": "numeric"},
    "cal_019": {"canonical": "china", "aliases": ["china"], "rule": "alias"},
    "cal_020": {"canonical": "300000", "aliases": ["300000", "300000.0"], "rule": "numeric"},
    "cal_021": {"canonical": "carbon dioxide", "aliases": ["carbon dioxide", "co2", "carbon-dioxide"], "rule": "alias"},
    "cal_022": {"canonical": "11", "aliases": ["11", "11.0", "eleven"], "rule": "numeric"},
    "cal_023": {"canonical": "mandarin", "aliases": ["mandarin", "mandarin chinese", "chinese"], "rule": "alias"},
    "cal_024": {"canonical": "diamond", "aliases": ["diamond"], "rule": "alias"},
    "cal_025": {"canonical": "8", "aliases": ["8", "8.0", "eight"], "rule": "numeric"},
    "cal_026": {"canonical": "mount everest", "aliases": ["mount everest", "everest", "mt. everest", "mt everest"], "rule": "alias"},
    "cal_027": {"canonical": "27", "aliases": ["27", "27.0", "twenty-seven"], "rule": "numeric"},
    "cal_028": {"canonical": "immanuel kant", "aliases": ["immanuel kant", "kant"], "rule": "alias"},
    "cal_029": {"canonical": "na", "aliases": ["na", "Na"], "rule": "alias"},
    "cal_030": {"canonical": "55", "aliases": ["55", "55.0"], "rule": "numeric"},
    "cal_031": {"canonical": "88", "aliases": ["88", "88.0"], "rule": "numeric"},
    "cal_032": {"canonical": "liver", "aliases": ["liver"], "rule": "alias"},
    "cal_033": {"canonical": "1991", "aliases": ["1991", "1991.0"], "rule": "numeric"},
    "cal_034": {"canonical": "iron", "aliases": ["iron", "fe"], "rule": "alias"},
    "cal_035": {"canonical": "6", "aliases": ["6", "6.0", "six"], "rule": "numeric"},
    "cal_036": {"canonical": "equilateral", "aliases": ["equilateral"], "rule": "alias"},
    "cal_037": {"canonical": "chimborazo", "aliases": ["chimborazo", "mount chimborazo", "mt. chimborazo", "mt chimborazo"], "rule": "alias"},
    "cal_038": {"canonical": "27", "aliases": ["27", "27.0", "twenty-seven"], "rule": "numeric"},
    "cal_039": {"canonical": "10", "aliases": ["10", "10.0", "ten"], "rule": "numeric"},
    "cal_040": {"canonical": "9", "aliases": ["9", "9.0", "nine"], "rule": "numeric"},
    "cal_041": {"canonical": "16", "aliases": ["16", "16.0", "sixteen"], "rule": "numeric"},
    "cal_042": {"canonical": "52", "aliases": ["52", "52.0"], "rule": "numeric"},
    "cal_043": {"canonical": "65536", "aliases": ["65536", "65536.0"], "rule": "numeric"},
    "cal_044": {"canonical": "12", "aliases": ["12", "12.0", "twelve"], "rule": "numeric"},
    "cal_045": {"canonical": "720", "aliases": ["720", "720.0"], "rule": "numeric"},
    "cal_046": {"canonical": "940000000", "aliases": ["940000000", "940000000.0"], "rule": "numeric"},
    "cal_047": {"canonical": "1", "aliases": ["1", "1.0", "one"], "rule": "numeric"},
    "cal_048": {"canonical": "8", "aliases": ["8", "8.0", "eight"], "rule": "numeric"},
    "cal_049": {"canonical": "128", "aliases": ["128", "128.0"], "rule": "numeric"},
    "cal_050": {"canonical": "8", "aliases": ["8", "8.0", "eight"], "rule": "numeric"},
    "cal_051": {"canonical": "5280", "aliases": ["5280", "5280.0"], "rule": "numeric"},
    "cal_052": {"canonical": "7", "aliases": ["7", "7.0", "seven"], "rule": "numeric"},
    "cal_053": {"canonical": "10080", "aliases": ["10080", "10080.0"], "rule": "numeric"},
    "cal_054": {"canonical": "880", "aliases": ["880", "880.0"], "rule": "numeric"},
    "cal_055": {"canonical": "101", "aliases": ["101", "101.0"], "rule": "numeric"},
    "cal_056": {"canonical": "2444", "aliases": ["2444", "2444.0"], "rule": "numeric"},
    "cal_057": {"canonical": "540", "aliases": ["540", "540.0"], "rule": "numeric"},
    "cal_058": {"canonical": "21", "aliases": ["21", "21.0", "twenty-one"], "rule": "numeric"},
    "cal_059": {"canonical": "8", "aliases": ["8", "8.0", "eight"], "rule": "numeric"},
    "cal_060": {"canonical": "10", "aliases": ["10", "10.0", "ten"], "rule": "numeric"},
    "cal_061": {"canonical": "15", "aliases": ["15", "15.0", "fifteen"], "rule": "numeric"},
    "cal_062": {"canonical": "72", "aliases": ["72", "72.0"], "rule": "numeric"},
    "cal_063": {"canonical": "26", "aliases": ["26", "26.0", "twenty-six"], "rule": "numeric"},
    "cal_064": {"canonical": "6561", "aliases": ["6561", "6561.0"], "rule": "numeric"},
    "cal_065": {"canonical": "8", "aliases": ["8", "8.0", "eight"], "rule": "numeric"},
    "cal_066": {"canonical": "144", "aliases": ["144", "144.0"], "rule": "numeric"},
    "cal_067": {"canonical": "nintendo", "aliases": ["nintendo"], "rule": "alias"},
    "cal_068": {"canonical": "71", "aliases": ["71", "71.0"], "rule": "numeric"},
    "cal_069": {"canonical": "nitrogen", "aliases": ["nitrogen", "n", "n2"], "rule": "alias"},
    "cal_070": {"canonical": "wellington", "aliases": ["wellington"], "rule": "alias"},
    "cal_071": {"canonical": "no", "aliases": ["no", "n"], "rule": "yes_no"},
    "cal_072": {"canonical": "no", "aliases": ["no", "n"], "rule": "yes_no"},
    "cal_073": {"canonical": "fruit", "aliases": ["fruit"], "rule": "alias"},
    "cal_074": {"canonical": "atlantic", "aliases": ["atlantic", "atlantic ocean", "the atlantic"], "rule": "alias"},
    "cal_075": {"canonical": "lithium", "aliases": ["lithium"], "rule": "alias"},
    "cal_076": {"canonical": "mauna kea", "aliases": ["mauna kea", "maunakea", "mauna-kea"], "rule": "alias"},
    "cal_077": {"canonical": "canada", "aliases": ["canada"], "rule": "alias"},
    "cal_078": {"canonical": "116", "aliases": ["116", "116.0"], "rule": "numeric"},
    "cal_079": {"canonical": "hydrogen", "aliases": ["hydrogen", "h", "h2"], "rule": "alias"},
    "cal_080": {"canonical": "no", "aliases": ["no", "n"], "rule": "yes_no"},
    "cal_081": {"canonical": "yes", "aliases": ["yes", "y"], "rule": "yes_no"},
    "cal_082": {"canonical": "no", "aliases": ["no", "n"], "rule": "yes_no"},
    "cal_083": {"canonical": "no", "aliases": ["no", "n"], "rule": "yes_no"},
    "cal_084": {"canonical": "no", "aliases": ["no", "n"], "rule": "yes_no"},
    "cal_085": {"canonical": "no", "aliases": ["no", "n"], "rule": "yes_no"},
    "cal_086": {"canonical": "no", "aliases": ["no", "n"], "rule": "yes_no"},
    "cal_087": {"canonical": "no", "aliases": ["no", "n"], "rule": "yes_no"},
    "cal_088": {"canonical": "japan", "aliases": ["japan"], "rule": "alias"},
    "cal_089": {"canonical": "no", "aliases": ["no", "n"], "rule": "yes_no"},
    "cal_090": {"canonical": "no", "aliases": ["no", "n"], "rule": "yes_no"},
    "cal_091": {"canonical": "yes", "aliases": ["yes", "y"], "rule": "yes_no"},
    "cal_092": {"canonical": "no", "aliases": ["no", "n"], "rule": "yes_no"},
    "cal_093": {"canonical": "no", "aliases": ["no", "n"], "rule": "yes_no"},
    "cal_094": {"canonical": "yes", "aliases": ["yes", "y"], "rule": "yes_no"},
    "cal_095": {"canonical": "yes", "aliases": ["yes", "y"], "rule": "yes_no"},
    "cal_096": {"canonical": "brazil", "aliases": ["brazil", "brasil"], "rule": "alias"},
    "cal_097": {"canonical": "0", "aliases": ["0", "0.0", "zero", "none"], "rule": "numeric"},
    "cal_098": {"canonical": "no", "aliases": ["no", "n"], "rule": "yes_no"},
    "cal_099": {"canonical": "no", "aliases": ["no", "n"], "rule": "yes_no"},
    "cal_100": {"canonical": "no", "aliases": ["no", "n"], "rule": "yes_no"}
}

# Parse CSV into list of dicts
import pandas as pd
dataset = pd.read_csv(io.StringIO(CALIBRATION_CSV))
print(f"Dataset loaded: {len(dataset)} items")
print(f"Difficulty distribution:\n{dataset['difficulty'].value_counts().to_string()}")
print(f"Answer key entries: {len(ANSWER_KEY)}")

In [ ]:
# Cell 4 — Scoring & Adjudication Functions

def normalize_text(x):
    """Normalize text for answer comparison."""
    if x is None:
        return None
    return " ".join(str(x).strip().lower().split())


def adjudicate(example_id, raw_answer, gold_answer):
    """Deterministic correctness grading with alias + numeric support.

    Grading hierarchy:
      1. Exact normalized canonical match
      2. Exact normalized alias match
      3. Numeric equivalence (if rule == 'numeric')
      4. Otherwise incorrect

    No LLM judge in the scoring loop.
    """
    spec = ANSWER_KEY.get(example_id)
    norm = normalize_text(raw_answer)
    if norm is None:
        return False

    # If no spec, fall back to exact match
    if spec is None:
        return norm == normalize_text(gold_answer)

    # Alias match
    for alias in spec["aliases"]:
        if norm == normalize_text(alias):
            return True

    # Numeric equivalence
    if spec["rule"] == "numeric":
        try:
            return float(norm) == float(spec["canonical"])
        except (ValueError, TypeError):
            pass

    return False


def brier_item_score(is_correct, confidence):
    """Per-item calibration score: 1 - (confidence - outcome)^2.

    This is an affine transform of per-item Brier loss.
    Strictly proper: expected score is uniquely maximized when
    stated confidence equals true probability of correctness.

    Range: [0, 1]. Higher is better.
    Reference: Brier (1950); Gneiting & Raftery (2007).
    """
    y = 1.0 if is_correct else 0.0
    return 1.0 - (confidence - y) ** 2


print("Scoring functions defined: adjudicate(), brier_item_score()")
print(f"  adjudicate('cal_001', '65', '65') -> {adjudicate('cal_001', '65', '65')}")
print(f"  adjudicate('cal_001', ' 65 ', '65') -> {adjudicate('cal_001', ' 65 ', '65')}")
print(f"  adjudicate('cal_017', 'Antarctica', 'antarctic desert') -> {adjudicate('cal_017', 'Antarctica', 'antarctic desert')}")
print(f"  brier_item_score(True, 0.9) -> {brier_item_score(True, 0.9):.4f}")
print(f"  brier_item_score(False, 0.9) -> {brier_item_score(False, 0.9):.4f}")
print(f"  brier_item_score(False, 0.2) -> {brier_item_score(False, 0.2):.4f}")

In [ ]:
# Cell 5 — Benchmark Task Definition

@kbench.task(name="metacog_calibration_v1")
def metacog_calibration(llm, example_id: str, prompt: str,
                        gold_answer: str, difficulty: str) -> float:
    """MetaJudge Confidence Calibration — single item evaluation.

    Workflow per item:
      1. Prompt the LLM with the calibration question
      2. Extract structured response (answer + confidence)
      3. Adjudicate correctness deterministically
      4. Compute per-item Brier-derived score
      5. Assert confidence is valid

    Returns the per-item calibration score (higher is better).
    """
    # Isolated chat so items don't leak context to each other
    with chats.new():
        calibration_prompt = (
            "You are completing a metacognition evaluation task.\n\n"
            "Task: Confidence Calibration\n"
            f"Question:\n{prompt}\n\n"
            "Instructions:\n"
            "1. Provide your best answer.\n"
            "2. Provide a confidence score from 0.0 to 1.0.\n"
            "3. Explain why you are or are not certain.\n"
            "4. Say whether you would verify this if possible.\n\n"
            "Return valid structured output with keys: "
            "answer, confidence, reason_for_uncertainty, would_verify_if_possible"
        )

        response = llm.prompt(calibration_prompt, schema=CalibrationResponse)

    # Validate confidence range
    conf = max(0.0, min(1.0, response.confidence))
    assertions.assert_true(
        0.0 <= response.confidence <= 1.0,
        expectation=f"Confidence {response.confidence} must be in [0, 1]"
    )

    # Adjudicate correctness deterministically
    is_correct = adjudicate(example_id, response.answer, gold_answer)

    # Compute Brier-derived score
    score = brier_item_score(is_correct, conf)

    print(f"  [{example_id}] answer={response.answer!r}, "
          f"conf={conf:.2f}, correct={is_correct}, score={score:.4f}")

    return score


# Quick single-item smoke test (pulls from embedded dataset)
print("=== Smoke test (single item) ===")
_smoke = dataset.iloc[0]
result = metacog_calibration.run(
    llm=kbench.llm,
    example_id=_smoke["example_id"],
    prompt=_smoke["prompt"],
    gold_answer=_smoke["gold_answer"],
    difficulty=_smoke["difficulty"],
)
print(f"Smoke test result: {result.result}")

In [ ]:
# Cell 6 — Batch Evaluation (full 100-item V2 calibration set)

@kbench.task(name="metacog_calibration_v1_batch")
def metacog_calibration_batch(llm, df) -> float:
    """Run the full calibration benchmark across all items.

    Returns the headline score: mean of per-item Brier-derived scores.
    This is the official MetaJudge calibration metric.
    """
    with kbench.client.enable_cache():
        runs = metacog_calibration.evaluate(
            stop_condition=lambda runs: len(runs) == df.shape[0],
            max_attempts=1,
            llm=[llm],
            evaluation_data=df,
            n_jobs=3,
        )

    eval_df = runs.as_dataframe()
    scores = eval_df["result"].astype(float)

    # Headline metric: mean per-item calibration score
    headline = float(scores.mean())

    # Diagnostics
    n_items = len(scores)
    min_score = float(scores.min())
    max_score = float(scores.max())

    print(f"\n{'='*50}")
    print(f"MetaJudge Calibration Results")
    print(f"{'='*50}")
    print(f"Items evaluated: {n_items}")
    print(f"Headline score (mean 1-Brier): {headline:.4f}")
    print(f"Score range: [{min_score:.4f}, {max_score:.4f}]")
    print(f"{'='*50}")

    return headline

# Run the full benchmark
headline_score = metacog_calibration_batch.run(kbench.llm, dataset)
print(f"\nFinal headline score: {headline_score.result}")

In [ ]:
%choose metacog_calibration_v1_batch